<a href="https://colab.research.google.com/github/brandonwolk/Kaneohe-Bay-Coral-Restoration-GIS/blob/main/capstone/03_Climate_Projections_and_Delta_Correction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# MODULE 3: PANGEO CMIP6 CLIMATE PROJECTIONS & DELTA BIAS CORRECTION
# Kāneʻohe Bay Coral Restoration Spatial Framework
# ==============================================================================

# 1. Automated Dependency Installer for Google Colab Environment
try:
    import rioxarray
    import intake_esm
except ImportError:
    print("Installing required geospatial and CMIP6 cloud libraries...")
    !pip install -q intake-esm gcsfs rioxarray xarray rasterio zarr cftime
    print("Dependencies installed successfully!")

import os
import shutil
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import xarray as xr
import rioxarray as rxr
import intake
import intake_esm
from google.colab import drive

# --- PROACTIVE MOUNT POINT CLEANSE & SMART DRIVE MOUNT ---
mount_point = '/content/drive'
try:
    drive.flush_and_unmount()
except:
    pass

if os.path.exists(mount_point):
    try:
        shutil.rmtree(mount_point)
    except Exception as e:
        print(f"Clearing mountpoint warning: {e}")

os.makedirs(mount_point, exist_ok=True)
drive.mount(mount_point, force_remount=True)

CAPSTONE_ROOT = "/content/drive/MyDrive/Capstone"
OUTPUT_DIR = f"{CAPSTONE_ROOT}/03 Predictive Rasters"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LAT_MIN, LAT_MAX = 18.5, 24.5
LON_MIN, LON_MAX = -160.0, -154.0
CMIP6_CATALOG_URL = "https://storage.googleapis.com/cmip6/pangeo-cmip6.json"
SOURCE_ID, MEMBER_ID, TABLE_ID, VARIABLE_ID, GRID_LABEL = "MIROC6", "r1i1p1f1", "Omon", "tos", "gn"

# Connect to Pangeo CMIP6 Catalog
col = intake.open_esm_datastore(CMIP6_CATALOG_URL)
experiment_query = dict(
    source_id=SOURCE_ID, member_id=MEMBER_ID, table_id=TABLE_ID,
    variable_id=VARIABLE_ID, grid_label=GRID_LABEL,
    experiment_id=["historical", "ssp245", "ssp585"],
)
cat_subset = col.search(**experiment_query)
dset_dict = cat_subset.to_dataset_dict(zarr_kwargs={"consolidated": True}, storage_options={"token": "anon"})

print(f"Loaded {len(dset_dict)} CMIP6 MIROC6 datasets from Pangeo cloud storage.")

Installing required geospatial and CMIP6 cloud libraries...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.3/405.3 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.4/38.4 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.0 MB/s eta 0:00:00

<div><progress max="3" value="3"></progress> 100.00% [3/3 00:04&lt;00:00]</div>

Loaded 3 CMIP6 MIROC6 datasets from Pangeo cloud storage.
